# Notes notebook

See relevant notes PDF

In [33]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


## Model definition 

We use a MLP with only 4 layers; this reflects the notes

In [34]:
class MLP(nn.Module):
    num_units: int
    
    def setup(self):
        # self.dense1 = nn.Dense(self.num_units)
        # self.dense2 = nn.Dense(self.num_units)
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f 
    

class SimpleMLP(nn.Module):
    """
    We ONLY work with "MLP" layers from above; see setup()

    Use sow to save intermediates for computation of preconditioner
    """
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = (
            *[
                MLP(self.num_units, name=f"layer_{i:02}") for i in range(self.num_layers - 1)
            ], 
            MLP(self.num_units, name=f"layer_{self.num_layers-1:02}") # For now, do all the same dimensions 
        )
        
    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}_output', x)
            
        return x


In [35]:
# Model definition 
L = 4 
n = 1 

n_samples = 1
X = jnp.linspace(0, 1, n_samples).reshape((-1, 1)) + .1

key = jax.random.PRNGKey(0)

model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init(key, jnp.ones((1, n)))['params']
params

{'layer_00': {'dense1': {'kernel': <jax.Array([[-1.5057027]], dtype=float32)>,
   'bias': <jax.Array([0.00563218], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[-0.73806745]], dtype=float32)>,
   'bias': <jax.Array([0.02606239], dtype=float32)>}},
 'layer_01': {'dense1': {'kernel': <jax.Array([[-0.4199463]], dtype=float32)>,
   'bias': <jax.Array([-0.00496414], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[0.9462289]], dtype=float32)>,
   'bias': <jax.Array([-0.01052106], dtype=float32)>}},
 'layer_02': {'dense1': {'kernel': <jax.Array([[-2.0070188]], dtype=float32)>,
   'bias': <jax.Array([0.00379716], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[-0.19703926]], dtype=float32)>,
   'bias': <jax.Array([0.00040974], dtype=float32)>}},
 'layer_03': {'dense1': {'kernel': <jax.Array([[1.5657363]], dtype=float32)>,
   'bias': <jax.Array([0.01429557], dtype=float32)>},
  'dense2': {'kernel': <jax.Array([[2.067297]], dtype=float32)>,
   'bias': <jax.Array([-0.00271632], dtype=float32)>}}}

We can look at the Jacobian easily. 

In [36]:
# Regular way
jacobian = jax.jacobian(model.apply, argnums=0)({'params': params}, X)

flattened, _ = jax.tree.flatten(jacobian)
reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
jacobian = aggregated_array.reshape((X.shape[0] * X.shape[1], -1))
treescope.display(jacobian)
treescope.display(jacobian.T @ jacobian)

In [37]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    # assert len(x) == 1
    # return jnp.squeeze(layer.apply(params, x))
    return layer.apply(params, x)

K_i = jax.jit(jax.jacrev(apply_layer, argnums=0))
M_i = jax.jit(jax.jacrev(apply_layer, argnums=1))


In [38]:

predictions, intermediates = model.apply({'params':params}, X, mutable=['intermediates'])


## Define the matrices 

This reflects the matrix structure of the constraints as in the notes

In [39]:
# %%timeit # Slightly better
# Initialize dgdu as a tridiagonal matrix with ones down the diagonal
dgdu = np.eye(n_samples * n * (L + 1)) # It's tridiagonal with ones down diagonal; will have to change once scaled up
counter = 0
for l in range(L):
    vals = -jnp.squeeze(M_i({'params': params[f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0]))

    # vals
    for s in range(n_samples):
        dgdu[counter * (n_samples * n) + s * n:counter * (n_samples * n) + s * n + n, 
             counter * (n_samples * n) + (n_samples * n) + s * n:counter * (n_samples * n) + (n_samples * n) + s * n + n] \
                =  vals


    counter += 1
dgdu = dgdu.T
dgduinv = np.linalg.inv(dgdu)

In [40]:
dgdu

array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-1.08828792,  1.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.39591346,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.38784218,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -3.23432617,  1.        ]])

In [41]:
dgduinv

array([[ 1.        , -0.        , -0.        , -0.        , -0.        ],
       [ 1.08828792,  1.        ,  0.        ,  0.        ,  0.        ],
       [-0.43086783, -0.39591346,  1.        ,  0.        ,  0.        ],
       [-0.16710872, -0.15355194,  0.38784218,  1.        , -0.        ],
       [-0.54048411, -0.49663706,  1.25440812,  3.23432617,  1.        ]])

In [42]:
# Number of trainable parameters
p = 16
dgdt = np.zeros((p, n_samples * n * (L + 1)))
counter = 0
for l in range(L): 
    flattened, _ = jax.tree.flatten(
            K_i({'params': params[f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    # Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
    reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
    
    layer_p = aggregated_array.shape[-1]

    for s in range(n_samples):
        dgdt[counter:counter + layer_p, 
            n_samples * n * l + n_samples * n + s * n:n_samples * n * l + n_samples * n + s * n + n] = aggregated_array[s, :].T
    counter += layer_p
    
dgdt = dgdt.T

In [43]:
dgdt

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.72277743, -0.07227774,  1.        , -0.14393164,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.94277161,
         0.12472269,  1.        , -0.06044659,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.19324292,  0.0130859 ,
         1.        ,  0.13880529,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  2.06569028, -0.05565041,  1.        ,
        -0.02787868]])

In [44]:
dgduinv @ dgdt

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [-0.72277743, -0.07227774,  1.        , -0.14393164,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.28615731,  0.02861573, -0.39591346,  0.05698447,  0.94277161,
         0.12472269,  1.        , -0.06044659,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.11098388,  0.01109839, -0.15355194,  0.02210098,  0.3656466 ,
         0.04837272,  0.38784218, -0.02344374, -0.19324292,  0.0130859 ,
         1.        ,  0.13880529,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.35895805,  0.0358958 , -0.49663706,  0.07148179,  1.18262037,
         0.15645315,  1.25440812, -0.07582469, -0.62501064,  0.04232407,
         3.23432617,  0.44894157,  2.06569028, -0.05565041,  1.        ,
        -0.02787868]])

In [45]:
jacobian

<jax.Array float32(1, 16) ≈0.54 ±0.97 [≥-0.63, ≤3.2] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>

In [46]:
np.linalg.norm((dgduinv @ dgdt)[-1, :] - jacobian) / np.linalg.norm(jacobian)

np.float64(1.5920159806682303e-08)

# JTJ matrix two ways 

This is the last page, with $K, M, E$

In [47]:
jtj = jacobian.T @ jacobian 
print(jnp.linalg.eigvalsh(jtj))
jtj

[-1.4104859e-07 -2.3744249e-08 -1.8017804e-08 -1.4070076e-08
 -9.5563442e-09 -8.7463272e-09 -4.6449307e-09 -4.0381654e-10
 -2.3287605e-10  4.0311318e-12  7.0838183e-11  8.5193748e-11
  1.0959201e-10  1.8471070e-08  1.8969805e-08  1.9710049e+01]


<jax.Array float32(16, 16) ≈0.29 ±1.2 [≥-2.0, ≤1e+01] nonzero:256
  <Arrayviz rendering>
| Device: GPU 0>

In [48]:
E = np.zeros((1, 5))
E[0, -1] = 1

dgdt.T @ dgduinv.T @ E.T @ E @ dgduinv @ dgdt 

array([[ 1.28850884e-01,  1.28850879e-02, -1.78271872e-01,
         2.56589632e-02,  4.24511107e-01,  5.61601200e-02,
         4.50279899e-01, -2.72178845e-02, -2.24352604e-01,
         1.51925674e-02,  1.16098743e+00,  1.61151191e-01,
         7.41496163e-01, -1.99761640e-02,  3.58958054e-01,
        -1.00072765e-02],
       [ 1.28850879e-02,  1.28850874e-03, -1.78271864e-02,
         2.56589622e-03,  4.24511090e-02,  5.61601177e-03,
         4.50279881e-02, -2.72178834e-03, -2.24352594e-02,
         1.51925668e-03,  1.16098738e-01,  1.61151184e-02,
         7.41496132e-02, -1.99761631e-03,  3.58958039e-02,
        -1.00072761e-03],
       [-1.78271872e-01, -1.78271864e-02,  2.46648367e-01,
        -3.55005044e-02, -5.87333101e-01, -7.77004344e-02,
        -6.22985559e-01,  3.76573528e-02,  3.10403446e-01,
        -2.10197038e-02, -1.60628623e+00, -2.22961018e-01,
        -1.02589834e+00,  2.76380574e-02, -4.96637057e-01,
         1.38455853e-02],
       [ 2.56589632e-02,  2.56589622e-03, -3.55005044e-02,
         5.10964590e-03,  8.45358177e-02,  1.11835511e-02,
         8.96673346e-02, -5.42008463e-03, -4.46768777e-02,
         3.02540049e-03,  2.31195415e-01,  3.20911455e-02,
         1.47659233e-01, -3.97799098e-03,  7.14817872e-02,
        -1.99281782e-03],
       [ 4.24511107e-01,  4.24511090e-02, -5.87333101e-01,
         8.45358177e-02,  1.39859094e+00,  1.85024688e-01,
         1.48348860e+00, -8.96718276e-02, -7.39150317e-01,
         5.00533127e-02,  3.82498002e+00,  5.30927442e-01,
         2.44292740e+00, -6.58133121e-02,  1.18262037e+00,
        -3.29698941e-02],
       [ 5.61601200e-02,  5.61601177e-03, -7.77004344e-02,
         1.11835511e-02,  1.85024688e-01,  2.44775897e-02,
         1.96256108e-01, -1.18630126e-02, -9.77848867e-02,
         6.62173498e-03,  5.06020534e-01,  7.02383244e-02,
         3.23183761e-01, -8.70668269e-03,  1.56453155e-01,
        -4.36170733e-03],
       [ 4.50279899e-01,  4.50279881e-02, -6.22985559e-01,
         8.96673346e-02,  1.48348860e+00,  1.96256108e-01,
         1.57353974e+00, -9.51151119e-02, -7.84018427e-01,
         5.30916629e-02,  4.05716503e+00,  5.63155948e-01,
         2.59121867e+00, -6.98083302e-02,  1.25440812e+00,
        -3.49712419e-02],
       [-2.72178845e-02, -2.72178834e-03,  3.76573528e-02,
        -5.42008463e-03, -8.96718276e-02, -1.18630126e-02,
        -9.51151119e-02,  5.74938419e-03,  4.73912405e-02,
        -3.20920999e-03, -2.45241792e-01, -3.40408568e-02,
        -1.56630333e-01,  4.21967553e-03, -7.58246938e-02,
         2.11389232e-03],
       [-2.24352604e-01, -2.24352594e-02,  3.10403446e-01,
        -4.46768777e-02, -7.39150317e-01, -9.77848867e-02,
        -7.84018427e-01,  4.73912405e-02,  3.90638302e-01,
        -2.64529970e-02, -2.02148828e+00, -2.80593257e-01,
        -1.29107841e+00,  3.47821004e-02, -6.25010642e-01,
         1.74244713e-02],
       [ 1.51925674e-02,  1.51925668e-03, -2.10197038e-02,
         3.02540049e-03,  5.00533127e-02,  6.62173498e-03,
         5.30916629e-02, -3.20920999e-03, -2.64529970e-02,
         1.79132728e-03,  1.36889862e-01,  1.90010363e-02,
         8.74284293e-02, -2.35535223e-03,  4.23240745e-02,
        -1.17993930e-03],
       [ 1.16098743e+00,  1.16098738e-01, -1.60628623e+00,
         2.31195415e-01,  3.82498002e+00,  5.06020534e-01,
         4.05716503e+00, -2.45241792e-01, -2.02148828e+00,
         1.36889862e-01,  1.04608658e+01,  1.45202346e+00,
         6.68111614e+00, -1.79991587e-01,  3.23432617e+00,
        -9.01687423e-02],
       [ 1.61151191e-01,  1.61151184e-02, -2.22961018e-01,
         3.20911455e-02,  5.30927442e-01,  7.02383244e-02,
         5.63155948e-01, -3.40408568e-02, -2.80593257e-01,
         1.90010363e-02,  1.45202346e+00,  2.01548530e-01,
         9.27374230e-01, -2.49837836e-02,  4.48941567e-01,
        -1.25158980e-02],
       [ 7.41496163e-01,  7.41496132e-02, -1.02589834e+00,
         1.47659233e-01,  2.44292740e+00,  3.23183761e-01,
         2.59121867e+00, -1.56

In [23]:
dgduinv.T @ E.T @ E @ dgduinv 

array([[ 0.29212307,  0.26842444, -0.67798766, -1.7481019 , -0.54048411],
       [ 0.26842444,  0.24664837, -0.62298556, -1.60628623, -0.49663706],
       [-0.67798766, -0.62298556,  1.57353974,  4.05716503,  1.25440812],
       [-1.7481019 , -1.60628623,  4.05716503, 10.4608658 ,  3.23432617],
       [-0.54048411, -0.49663706,  1.25440812,  3.23432617,  1.        ]])

In [24]:
E @ dgduinv 

array([[-0.54048411, -0.49663706,  1.25440812,  3.23432617,  1.        ]])

In [32]:
dgdt.T @ dgduinv.T @ dgduinv @ dgdt 

array([[ 7.45461519e-01,  7.45461488e-02, -1.03138462e+00,
         1.48448882e-01,  7.34872972e-01,  9.72190212e-02,
         7.79481437e-01, -4.71169950e-02, -2.45799452e-01,
         1.66448915e-02,  1.27197130e+00,  1.76556340e-01,
         7.41496163e-01, -1.99761640e-02,  3.58958054e-01,
        -1.00072765e-02],
       [ 7.45461488e-02,  7.45461457e-03, -1.03138457e-01,
         1.48448875e-02,  7.34872942e-02,  9.72190172e-03,
         7.79481405e-02, -4.71169931e-03, -2.45799442e-02,
         1.66448908e-03,  1.27197125e-01,  1.76556332e-02,
         7.41496132e-02, -1.99761631e-03,  3.58958039e-02,
        -1.00072761e-03],
       [-1.03138462e+00, -1.03138457e-01,  1.42697403e+00,
        -2.05386716e-01, -1.01673481e+00, -1.34507551e-01,
        -1.07845294e+00,  6.51888026e-02,  3.40076271e-01,
        -2.30290694e-02, -1.75983817e+00, -2.44274839e-01,
        -1.02589834e+00,  2.76380574e-02, -4.96637057e-01,
         1.38455853e-02],
       [ 1.48448882e-01,  1.48448875e-02, -2.05386716e-01,
         2.95616472e-02,  1.46340312e-01,  1.93598926e-02,
         1.55223502e-01, -9.38273142e-03, -4.89477362e-02,
         3.31461178e-03,  2.53296398e-01,  3.51588788e-02,
         1.47659233e-01, -3.97799098e-03,  7.14817872e-02,
        -1.99281782e-03],
       [ 7.34872972e-01,  7.34872942e-02, -1.01673481e+00,
         1.46340312e-01,  2.42110669e+00,  3.20297020e-01,
         2.56807339e+00, -1.55231280e-01, -8.09808935e-01,
         5.48381282e-02,  4.19062662e+00,  5.81681123e-01,
         2.44292740e+00, -6.58133121e-02,  1.18262037e+00,
        -3.29698941e-02],
       [ 9.72190212e-02,  9.72190172e-03, -1.34507551e-01,
         1.93598926e-02,  3.20297020e-01,  4.23732590e-02,
         3.39739779e-01, -2.05361112e-02, -1.07132573e-01,
         7.25473564e-03,  5.54393254e-01,  7.69527137e-02,
         3.23183761e-01, -8.70668269e-03,  1.56453155e-01,
        -4.36170733e-03],
       [ 7.79481437e-01,  7.79481405e-02, -1.07845294e+00,
         1.55223502e-01,  2.56807339e+00,  3.39739779e-01,
         2.72396130e+00, -1.64654172e-01, -8.58966184e-01,
         5.81669276e-02,  4.44500721e+00,  6.16990493e-01,
         2.59121867e+00, -6.98083302e-02,  1.25440812e+00,
        -3.49712419e-02],
       [-4.71169950e-02, -4.71169931e-03,  6.51888026e-02,
        -9.38273142e-03, -1.55231280e-01, -2.05361112e-02,
        -1.64654172e-01,  9.95278328e-03,  5.19215769e-02,
        -3.51599243e-03, -2.68685529e-01, -3.72949715e-02,
        -1.56630333e-01,  4.21967553e-03, -7.58246938e-02,
         2.11389232e-03],
       [-2.45799452e-01, -2.45799442e-02,  3.40076271e-01,
        -4.89477362e-02, -8.09808935e-01, -1.07132573e-01,
        -8.58966184e-01,  5.19215769e-02,  4.27981130e-01,
        -2.89817549e-02, -2.21473120e+00, -3.07416396e-01,
        -1.29107841e+00,  3.47821004e-02, -6.25010642e-01,
         1.74244713e-02],
       [ 1.66448915e-02,  1.66448908e-03, -2.30290694e-02,
         3.31461178e-03,  5.48381282e-02,  7.25473564e-03,
         5.81669276e-02, -3.51599243e-03, -2.89817549e-02,
         1.96256811e-03,  1.49975764e-01,  2.08174286e-02,
         8.74284293e-02, -2.35535223e-03,  4.23240745e-02,
        -1.17993930e-03],
       [ 1.27197130e+00,  1.27197125e-01, -1.75983817e+00,
         2.53296398e-01,  4.19062662e+00,  5.54393254e-01,
         4.44500721e+00, -2.68685529e-01, -2.21473120e+00,
         1.49975764e-01,  1.14608658e+01,  1.59082874e+00,
         6.68111614e+00, -1.79991587e-01,  3.23432617e+00,
        -9.01687423e-02],
       [ 1.76556340e-01,  1.76556332e-02, -2.44274839e-01,
         3.51588788e-02,  5.81681123e-01,  7.69527137e-02,
         6.16990493e-01, -3.72949715e-02, -3.07416396e-01,
         2.08174286e-02,  1.59082874e+00,  2.20815437e-01,
         9.27374230e-01, -2.49837836e-02,  4.48941567e-01,
        -1.25158980e-02],
       [ 7.41496163e-01,  7.41496132e-02, -1.02589834e+00,
         1.47659233e-01,  2.44292740e+00,  3.23183761e-01,
         2.59121867e+00, -1.56

In [28]:
jnp.linalg.eigvalsh(dgdt.T @ dgduinv.T @ E.T @ E @ dgduinv @ dgdt  ) # rank of 1, only 1 sample

<jax.Array float64(16,) ≈1.2 ±4.8 [≥-4.4e-16, ≤2e+01] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>

In [31]:
jnp.linalg.eigvalsh(dgdt.T @ dgduinv.T @ dgduinv @ dgdt ) # rank of 4 or 5 basically

<jax.Array float64(16,) ≈1.5 ±5.1 [≥-1.5e-17, ≤2.1e+01] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>

But dominating eigenvalues the same... the real issue is actually computation of the matrices; these are pretty darn large! 

## Woodbury identity 

Hmm, the good news here is that 


In [54]:
lamb = 1
jtjinv = jnp.linalg.inv(lamb * jnp.eye(jtj.shape[0]) + jtj)

In [57]:
jtjinv

<jax.Array float64(16, 16) ≈0.049 ±0.24 [≥-0.32, ≤1.0] nonzero:256
  <Arrayviz rendering>
| Device: GPU 0>

In [58]:
jacobian @ jacobian.T

<jax.Array([[19.710049]], dtype=float32)>

In [60]:
jnp.eye(jtj.shape[0]) - jacobian.T @ jnp.linalg.inv(1 + jacobian @ jacobian.T) @ jacobian

<jax.Array float64(16, 16) ≈0.049 ±0.24 [≥-0.32, ≤1.0] nonzero:256
  <Arrayviz rendering>
| Device: GPU 0>

In [62]:
1 + jacobian @ jacobian.T

<jax.Array([[20.710049]], dtype=float32)>

In [67]:
1 + E @ dgduinv @ dgdt @ dgdt.T @ dgduinv.T @ E.T 

array([[20.71004954]])

In [73]:
# Inside is diaognal!!
dgdt @ dgdt.T

array([[0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 1.5483476 , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 1.90802785, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 1.05678098, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 5.27095052]])

In [74]:
dgduinv @  dgduinv.T

array([[ 1.        ,  1.08828792, -0.43086783, -0.16710872, -0.54048411],
       [ 1.08828792,  2.1843706 , -0.86482172, -0.33541434, -1.08483939],
       [-0.43086783, -0.86482172,  1.34239456,  0.52063723,  1.68391064],
       [-0.16710872, -0.33541434,  0.52063723,  1.20192508,  3.88741775],
       [-0.54048411, -1.08483939,  1.68391064,  3.88741775, 13.57317698]])

## 